<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the core libraries for data manipulation and time series decomposition used in this notebook.

In [ ]:
!pip install pandas statsmodels openbb

The openbb_terminal package requires a separate installation process with additional dependencies. Follow the official OpenBB installation guide at https://docs.openbb.co for setup instructions.

## Imports and setup

pandas handles tabular data and time series indexing. statsmodels provides two decomposition methods: seasonal_decompose for classical additive decomposition and STL for a more robust, flexible alternative. openbb provides a convenient API for pulling economic data like US unemployment directly into a DataFrame.

In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from openbb import obb
obb.user.preferences.output_type = "dataframe"

## Fetch and prepare unemployment data

We pull US unemployment data starting from 2010 using OpenBB, then filter it to end before 2020 so we have a clean pre-pandemic window without the extreme COVID-era distortions.

In [ ]:
df = obb.economy.unemployment(start_date="2010-01-01")

Choosing a stable date range matters more than most beginners realize. The pandemic caused unemployment to spike from around 3.5% to nearly 15% in a single month, which would dominate any decomposition and obscure the seasonal and trend patterns we want to study. Working with 2010-2019 gives us a full business cycle with a clear downward trend and consistent seasonality.

## Visualize raw data with rolling statistics

A 12-month rolling mean and rolling standard deviation help us visually separate the long-term trend from short-term volatility before we run any formal decomposition.

In [ ]:
df["rolling_mean"] = df["value"].rolling(window=12).mean()
df["rolling_std"] = df["value"].rolling(window=12).std()
df.plot(title="Unemployment rate")

The rolling mean smooths out monthly noise and reveals the underlying direction of unemployment over time. The rolling standard deviation shows whether the data's variability is stable or changing. If the standard deviation shrinks as the mean falls, that tells us the series is becoming more predictable, which is useful information before we choose a decomposition model. This kind of quick visual check is how professionals decide whether a simple additive model is appropriate or whether they need something more flexible.

## Decompose with classical additive method

Classical additive decomposition assumes the original series equals trend plus seasonality plus residual noise. This is the simplest approach and a good starting point for data where seasonal swings stay roughly constant in size over time.

In [ ]:
decomposition_results = seasonal_decompose(
    df["value"],
    model="additive",
    period=12,
).plot()

The plot shows four panels: the original series, the extracted trend, the seasonal pattern, and the residuals. If the residuals look random with no visible structure, the decomposition captured the meaningful patterns well. If you see leftover trends or cycles in the residuals, that signals the additive model missed something. One limitation of seasonal_decompose is that it uses a fixed seasonal pattern across the entire series, which can be too rigid for real-world economic data where seasonal effects shift over time.

## Decompose with STL for comparison

STL (Seasonal and Trend decomposition using Loess) fits the seasonal component locally rather than assuming it stays fixed, making it more adaptive to changes in the data's structure over time.

In [ ]:
stl_decomposition = STL(df[["value"]], period=12).fit()
stl_decomposition.plot().suptitle("STL Decomposition")

Comparing the STL output to the classical decomposition above is where the real learning happens. Look at the seasonal panels side by side. If the seasonal pattern in STL shifts slightly across years while the classical version repeats identically, that tells you the seasonal effect in unemployment isn't perfectly stable. STL's flexibility is why macro research desks tend to prefer it for economic indicators. The residuals from STL are also typically smaller and more random, which means the model is capturing more of the actual signal and leaving less structure on the table.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.